# Financial Stress Index & Credit Pricing System
## Notebook 02 — Feature Engineering

Objective:
- Clean raw decision-time features
- Convert messy variables into usable numeric form
- Prepare features for:
  - Financial Stress Index (FSI)
  - Regression-based interest rate pricing

Important:
- No model training in this notebook
- No target leakage

In [1]:
import pandas as pd
import numpy as np

### Load Dataset from Notebook 01

This dataset already:
- Contains only decision-time features
- Preserves issue date


In [5]:
DATA_PATH = "../data/processed/modeling_dataset.csv"
df = pd.read_csv(DATA_PATH)

df.shape

(2260701, 10)

### Features to Engineer

We focus on:
- Employment length (string → numeric)
- Credit utilization (percentage → numeric)
- Loan term (string → numeric)
- Conservative missing value handling


### Employment Length Cleaning

Original values:
- "10+ years"
- "< 1 year"
- "3 years"
- Missing values

Strategy:
- Convert to numeric years
- Cap at 10
- Missing → median

In [6]:
def clean_emp_length(val):
    if pd.isna(val):
        return np.nan
    if val == "10+ years":
        return 10
    if val == "< 1 year":
        return 0
    return int(val.split()[0])

df["emp_length"] = df["emp_length"].apply(clean_emp_length)
df["emp_length"].fillna(df["emp_length"].median(), inplace=True)

df["emp_length"].describe()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_12448\4238647469.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["emp_length"].fillna(df["emp_length"].median(), inplace=True)


count    2.260701e+06
mean     5.935821e+00
std      3.597319e+00
min      0.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.000000e+01
max      1.000000e+01
Name: emp_length, dtype: float64

### Credit Utilization Cleaning

Original:
- Stored as percentage string
- Some missing values

Strategy:
- Convert to float
- Cap between 0–100
- Missing → median


In [7]:
df["revol_util"] = df["revol_util"].replace("%", "").astype(float)
df["revol_util"] = df["revol_util"].clip(0, 100)
df["revol_util"].fillna(df["revol_util"].median(), inplace=True)

df["revol_util"].describe()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_12448\723435369.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["revol_util"].fillna(df["revol_util"].median(), inplace=True)


count    2.260701e+06
mean     5.032695e+01
std      2.467195e+01
min      0.000000e+00
25%      3.150000e+01
50%      5.030000e+01
75%      6.930000e+01
max      1.000000e+02
Name: revol_util, dtype: float64

### Loan Term Normalization

Original:
- "36 months"
- "60 months"

Strategy:
- Convert to integer months

In [8]:
df["term"] = df["term"].str.extract(r"(\d+)").astype("Int64")
print(df["term"].value_counts())

term
36    1609754
60     650914
Name: count, dtype: Int64


### Debt-to-Income Ratio

Strategy:
- Ensure non-negative
- Cap extreme values conservatively

In [9]:
df["dti"] = df["dti"].clip(0, 50)
df["dti"].describe()

count    2.258957e+06
mean     1.853480e+01
std      9.012037e+00
min      0.000000e+00
25%      1.189000e+01
50%      1.784000e+01
75%      2.449000e+01
max      5.000000e+01
Name: dti, dtype: float64

### Delinquencies & Credit Inquiries

These are sparse but high-impact variables.
Strategy:
- Replace missing with 0
- No scaling yet

In [10]:
df["delinq_2yrs"].fillna(0, inplace=True)
df["inq_last_6mths"].fillna(0, inplace=True)

df[["delinq_2yrs", "inq_last_6mths"]].describe()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_12448\1154284957.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["delinq_2yrs"].fillna(0, inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_12448\1154284957.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, whe

,delinq_2yrs,inq_last_6mths
count,2.260701e+06,2.260701e+06
mean,3.068707e-01,5.768193e-01
std,8.672199e-01,8.859560e-01
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00
75%,0.000000e+00,1.000000e+00
max,5.800000e+01,3.300000e+01


### Final Feature Sanity Check

Ensure:
- No missing values
- Correct data types
- No leakage

In [11]:
df.isna().sum()

annual_inc          37
emp_length           0
loan_amnt           33
term                33
dti               1744
revol_util           0
delinq_2yrs          0
inq_last_6mths       0
int_rate            33
issue_d             33
dtype: int64

### Final Missing Value Handling

Very small number of rows contain missing values.
We safely drop them to preserve data integrity.

In [12]:
df.shape

(2260701, 10)

In [17]:
df = df.dropna()
df.isnull().sum()

annual_inc        0
emp_length        0
loan_amnt         0
term              0
dti               0
revol_util        0
delinq_2yrs       0
inq_last_6mths    0
int_rate          0
issue_d           0
dtype: int64

In [19]:
df.isnull().sum()

annual_inc        0
emp_length        0
loan_amnt         0
term              0
dti               0
revol_util        0
delinq_2yrs       0
inq_last_6mths    0
int_rate          0
issue_d           0
dtype: int64

In [21]:
df.shape

(2258953, 10)

In [22]:
df.dtypes

annual_inc        float64
emp_length        float64
loan_amnt         float64
term                Int64
dti               float64
revol_util        float64
delinq_2yrs       float64
inq_last_6mths    float64
int_rate          float64
issue_d            object
dtype: object

### Saving Feature-Engineered Dataset

This dataset is now ready for:
- Financial Stress Index design
- Regression modeling

In [23]:
OUTPUT_PATH = "../data/processed/modeling_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False)

### Notebook 02 Summary

- Cleaned all decision-time features
- Converted categorical variables to numeric
- Handled missing values conservatively
- Prepared dataset for FSI engineering

Next Notebook:
03_fsi_design_engine.ipynb